In [1]:
# ============================================================
# CELL 1 — SETUP RAG + FINE-TUNED MODEL STAGE
# Extract LoRA adapter and load existing test/retrieval files
# ============================================================

!pip install -q \
    "transformers>=4.48,<5" \
    "peft==0.20.0" \
    accelerate bitsandbytes \
    bert-score==0.3.13 \
    sacrebleu rapidfuzz

from google.colab import drive
drive.mount("/content/drive")

import os, glob, json, zipfile, shutil
import pandas as pd
import numpy as np
import torch
from pathlib import Path

# ---------- Folders ----------
RAG_DIR = Path("/content/drive/MyDrive/Govt_Chatbot/RAG")
OUT = Path("/content/drive/MyDrive/Govt_Chatbot/RAG+Fine_Tuning")
OUT.mkdir(parents=True, exist_ok=True)

# ---------- Find uploaded fine-tuned ZIP ----------
zip_files = glob.glob("/content/Fine_Tuned_Qwen*.zip")

if not zip_files:
    raise FileNotFoundError("Fine_Tuned_Qwen.zip not found in /content")

ZIP_PATH = zip_files[0]
EXTRACT_DIR = Path("/content/Fine_Tuned_Qwen")

# Extract adapter
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

# Find adapter_config.json automatically
adapter_configs = list(EXTRACT_DIR.rglob("adapter_config.json"))

if not adapter_configs:
    raise FileNotFoundError("adapter_config.json not found inside ZIP")

ADAPTER_DIR = adapter_configs[0].parent

with open(ADAPTER_DIR / "adapter_config.json", encoding="utf-8") as f:
    adapter_info = json.load(f)

# ---------- Load already stored RAG files ----------
TEST_PATH = RAG_DIR / "test_questions.csv"
RETRIEVAL_JSON = RAG_DIR / "retrievals.json"
RETRIEVAL_CSV = RAG_DIR / "retrievals.csv"

tests = pd.read_csv(TEST_PATH).fillna("")

with open(RETRIEVAL_JSON, encoding="utf-8") as f:
    retrievals = json.load(f)

retrieval_meta = pd.read_csv(RETRIEVAL_CSV).fillna("")

# Safety checks
assert len(tests) == len(retrievals), "Test/retrieval count mismatch"
assert len(tests) == len(retrieval_meta), "Retrieval CSV count mismatch"

for i in range(len(tests)):
    assert str(tests.iloc[i]["question"]).strip() == \
           str(retrieval_meta.iloc[i]["question"]).strip(), \
           f"Question/retrieval mismatch at row {i}"

# Save run information
run_info = {
    "stage": "RAG + Fine-Tuning",
    "base_model": adapter_info["base_model_name_or_path"],
    "lora_r": adapter_info["r"],
    "lora_alpha": adapter_info["lora_alpha"],
    "lora_dropout": adapter_info["lora_dropout"],
    "target_modules": adapter_info["target_modules"],
    "max_new_tokens": 500,
    "test_questions": len(tests),
    "retrieval_source": str(RETRIEVAL_JSON)
}

with open(OUT / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)

# Keep copies of exactly what this experiment used
shutil.copy2(TEST_PATH, OUT / "test_questions_used.csv")
shutil.copy2(RETRIEVAL_CSV, OUT / "retrievals_used.csv")
shutil.copy2(RETRIEVAL_JSON, OUT / "retrievals_used.json")

print("Adapter folder:", ADAPTER_DIR)
print("Original fine-tuned base:", adapter_info["base_model_name_or_path"])
print("LoRA r:", adapter_info["r"])
print("LoRA alpha:", adapter_info["lora_alpha"])
print("Target modules:", adapter_info["target_modules"])
print("Test questions:", len(tests))
print("Retrieval sets:", len(retrievals))
print("Output:", OUT)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible

In [2]:
# ============================================================
# CELL 2 — RAG + FINE-TUNED QWEN INFERENCE
# Existing top-5 retrievals -> LoRA Qwen2.5-7B -> predictions
# max_new_tokens = 500
# ============================================================

import gc
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel


if not torch.cuda.is_available():
    raise RuntimeError("Please use a T4 GPU runtime.")


# ------------------------------------------------------------
# Load tokenizer saved with the fine-tuned adapter
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR
)

# Safer padding for causal generation
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"


# ------------------------------------------------------------
# Load Qwen2.5-7B base in 4-bit
# Then attach your friend's trained LoRA
# ------------------------------------------------------------

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Qwen base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("Attaching fine-tuned LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False
)

model.eval()

device = model.get_input_embeddings().weight.device

MAX_NEW_TOKENS = 500


# ------------------------------------------------------------
# Build prompt using the already retrieved top-5 contexts
# ------------------------------------------------------------

def generate_answer(question, docs):

    context = "\n\n".join([
        f"[Context {i}]\n"
        f"শিরোনাম: {doc.get('title', '')}\n"
        f"তথ্য: {doc.get('text', '')}"
        for i, doc in enumerate(docs, 1)
    ])

    messages = [
        {
            "role": "system",
            "content":
                "তুমি বাংলাদেশের সরকারি সেবা বিষয়ক সহকারী। "
                "শুধু প্রদত্ত Context ব্যবহার করে প্রশ্নের সঠিক ও সংক্ষিপ্ত উত্তর বাংলায় দাও। "
                "প্রশ্ন ও Context-এর ভাষা আলাদা হলেও সমার্থক তথ্য বুঝে উত্তর দাও। "
                "ফি, সংখ্যা, সময় ও প্রয়োজনীয় কাগজপত্র নির্ভুলভাবে উল্লেখ করো। "
                "Context-এ উত্তর একেবারেই না থাকলে শুধু বলো: "
                "'প্রদত্ত তথ্যে এই প্রশ্নের উত্তর পাওয়া যায়নি।'"
        },
        {
            "role": "user",
            "content":
                f"Context:\n{context}\n\n"
                f"প্রশ্ন: {question}\n\n"
                "উত্তর:"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=7000
    ).to(device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    truncated = (
        len(generated) >= MAX_NEW_TOKENS
        and
        generated[-1].item() != tokenizer.eos_token_id
    )

    if not answer:
        answer = "প্রদত্ত তথ্যে এই প্রশ্নের উত্তর পাওয়া যায়নি।"

    return answer, truncated


# ------------------------------------------------------------
# Resume support
# ------------------------------------------------------------

PARTIAL = OUT / "predictions_partial.csv"

done = {}

if PARTIAL.exists():

    old = pd.read_csv(PARTIAL).fillna("")

    for _, row in old.iterrows():
        done[(str(row["domain"]), str(row["id"]))] = row.to_dict()

print("Already completed:", len(done))


# ------------------------------------------------------------
# Generate answers
# ------------------------------------------------------------

predictions = []

for i, row in tqdm(
    tests.iterrows(),
    total=len(tests),
    desc="RAG + Fine-Tuned Qwen"
):

    key = (
        str(row["domain"]),
        str(row["id"])
    )

    if key in done:

        result = done[key]

    else:

        answer, truncated = generate_answer(
            row["question"],
            retrievals[i]
        )

        result = {
            "id": str(row["id"]),
            "domain": str(row["domain"]),
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "truncated": truncated
        }

        done[key] = result

    predictions.append(result)

    # Checkpoint after every answer
    pd.DataFrame(predictions).to_csv(
        PARTIAL,
        index=False,
        encoding="utf-8-sig"
    )


# ---------- Final prediction file ----------
pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nInference complete.")
print("Predictions:", len(pred_df))
print(
    "Truncated:",
    pred_df["truncated"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)
print("Saved:", OUT / "predictions.csv")


# Free GPU before BERTScore
del model, base_model, tokenizer

gc.collect()
torch.cuda.empty_cache()

Loading Qwen base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Attaching fine-tuned LoRA adapter...
Already completed: 0


RAG + Fine-Tuned Qwen:   0%|          | 0/248 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Inference complete.
Predictions: 248
Truncated: 0
Saved: /content/drive/MyDrive/Govt_Chatbot/RAG+Fine_Tuning/predictions.csv


In [3]:
# ============================================================
# CELL 3 — FINAL RAG + FINE-TUNING EVALUATION
# Exact Match, Fuzzy Match, Corpus BLEU,
# ROUGE-1/2/L, Token F1, BERT Precision/Recall/F1
# ============================================================

import re
import unicodedata
from collections import Counter

import pandas as pd
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score


df = pd.read_csv(
    OUT / "predictions.csv"
).fillna("")

assert (df["gold"].astype(str).str.strip() != "").all(), \
       "Blank gold answer found."


# ------------------------------------------------------------
# Normalization
# ------------------------------------------------------------

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)

def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(BN_TO_EN).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ------------------------------------------------------------
# Exact Match
# ------------------------------------------------------------

def exact_match(pred, gold):
    return float(
        normalize(pred) == normalize(gold)
    )


# ------------------------------------------------------------
# Token F1
# ------------------------------------------------------------

def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum(
        (Counter(p) & Counter(g)).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# ROUGE-N
# ------------------------------------------------------------

def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    p_ngram = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    g_ngram = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum(
        (p_ngram & g_ngram).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(p_ngram.values())
    recall = overlap / sum(g_ngram.values())

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# ROUGE-L
# ------------------------------------------------------------

def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(dp[j-1] + 1)

            else:
                new.append(
                    max(dp[j], new[-1])
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# Row-level lexical metrics
# ------------------------------------------------------------

df["Exact Match"] = [
    exact_match(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100
    for p, g in zip(df["prediction"], df["gold"])
]

df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(df["prediction"], df["gold"])
]

df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(df["prediction"], df["gold"])
]


# ------------------------------------------------------------
# Corpus BLEU
# ------------------------------------------------------------

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_bleu = [
    " ".join(tokens(x))
    for x in df["prediction"]
]

gold_bleu = [
    " ".join(tokens(x))
    for x in df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_bleu,
        [gold_bleu]
    ).score
    / 100
)


# ------------------------------------------------------------
# Multilingual BERTScore
# ------------------------------------------------------------

print("Calculating BERTScore...")

P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(),
    df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=4,
    device="cpu",
    idf=False,
    rescale_with_baseline=False,
    verbose=True
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()


# ------------------------------------------------------------
# Final result.csv
# ------------------------------------------------------------

result = pd.DataFrame({

    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1"
    ],

    "score": [
        df["Exact Match"].mean(),
        df["Fuzzy Match"].mean(),
        corpus_bleu,
        df["ROUGE-1"].mean(),
        df["ROUGE-2"].mean(),
        df["ROUGE-L"].mean(),
        df["Token F1"].mean(),
        df["BERT Precision"].mean(),
        df["BERT Recall"].mean(),
        df["BERT F1"].mean()
    ]
})


# Store row metrics
df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

result.to_csv(
    OUT / "result.csv",
    index=False,
    encoding="utf-8-sig"
)

display(result)

print("\nSaved:")
print(OUT / "run_config.json")
print(OUT / "retrievals_used.csv")
print(OUT / "predictions.csv")
print(OUT / "result.csv")

Calculating BERTScore...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

computing greedy matching.


  0%|          | 0/62 [00:00<?, ?it/s]

done in 36.05 seconds, 6.88 sentences/sec


,metric,score
0,Exact Match,0.391129
1,Fuzzy Match,0.923815
2,Corpus BLEU,0.636305
3,ROUGE-1,0.788229
4,ROUGE-2,0.723988
5,ROUGE-L,0.771200
6,Token F1,0.788229
7,BERT Precision,0.926471
8,BERT Recall,0.921204
9,BERT F1,0.922711



Saved:
/content/drive/MyDrive/Govt_Chatbot/RAG+Fine_Tuning/run_config.json
/content/drive/MyDrive/Govt_Chatbot/RAG+Fine_Tuning/retrievals_used.csv
/content/drive/MyDrive/Govt_Chatbot/RAG+Fine_Tuning/predictions.csv
/content/drive/MyDrive/Govt_Chatbot/RAG+Fine_Tuning/result.csv
